In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import LabelEncoder
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [ ]:
path_root = r'D:\Materials\BTL_HGY\data'
path_train = os.path.join(path_root, 'logs_train.parquet')
path_val = os.path.join(path_root, 'logs_val.parquet')
path_metadata_train = os.path.join(path_root, 'metadata_train.parquet')
path_metadata_val = os.path.join(path_root, 'metadata_val.parquet')
path_metadata_test = os.path.join(path_root, 'metadata_test.parquet')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
print("Loading data...")
df_train = pd.read_parquet(path_train)
df_val = pd.read_parquet(path_val)
df_metadata_train = pd.read_parquet(path_metadata_train)
df_metadata_val = pd.read_parquet(path_metadata_val)
df_metadata_test = pd.read_parquet(path_metadata_test)

In [ ]:
print(f"Train logs: {len(df_train)}")
print(f"Val logs: {len(df_val)}")
print(f"Metadata train: {len(df_metadata_train)}")
print(f"Metadata val: {len(df_metadata_val)}")
print(f"Metadata test: {len(df_metadata_test)}")
print("\nPreprocessing...")
df_train_clean = df_train[df_train['tv_show_id'] != 0].copy()
df_val_clean = df_val[df_val['tv_show_id'] != 0].copy()
df_metadata_train_clean = df_metadata_train[df_metadata_train['tv_show_id'] != 0].copy()
df_metadata_val_clean = df_metadata_val[df_metadata_val['tv_show_id'] != 0].copy()
df_metadata_test_clean = df_metadata_test[df_metadata_test['tv_show_id'] != 0].copy()

In [ ]:
print(f"Train logs after filter: {len(df_train_clean)}")
print(f"Val logs after filter: {len(df_val_clean)}")

df_merged = pd.concat([df_train_clean, df_val_clean], ignore_index=True)
df_merged_metadata = pd.concat([df_metadata_train_clean, df_metadata_val_clean, df_metadata_test_clean], ignore_index=True)

df_merged['user_id'] = df_merged['user_id'].astype(str).str.strip()
df_merged['tv_show_id'] = df_merged['tv_show_id'].astype(str).str.strip()

print(f"Merged logs: {len(df_merged)}")
print(f"Merged metadata: {len(df_merged_metadata)}")

In [ ]:
# Encode users and items
user_enc = LabelEncoder()
item_enc = LabelEncoder()

df_merged['user_idx'] = user_enc.fit_transform(df_merged['user_id'])
df_merged['item_idx'] = item_enc.fit_transform(df_merged['tv_show_id'])

# Build mappings
useridx_to_userid = dict(zip(range(len(user_enc.classes_)), user_enc.classes_))
userid_to_useridx = dict(zip(user_enc.classes_, range(len(user_enc.classes_))))

itemidx_to_showid = dict(zip(range(len(item_enc.classes_)), item_enc.classes_))
showid_to_itemidx = dict(zip(item_enc.classes_, range(len(item_enc.classes_))))

n_users = len(user_enc.classes_)
n_items = len(item_enc.classes_)

print(f"✓ Total unique users: {n_users}")
print(f"✓ Total unique items: {n_items}")
print(f"✓ Mappings created")


In [ ]:
class InteractionDataset(Dataset):
    """PyTorch Dataset for user-item interactions"""
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return (
            torch.tensor(row.user_idx, dtype=torch.long),
            torch.tensor(row.item_idx, dtype=torch.long),
            torch.tensor(row.label, dtype=torch.float),
        )
    
class NeuMF(nn.Module):
    """Neural Matrix Factorization Model"""
    def __init__(
        self,
        num_users: int,
        num_items: int,
        emb_dim: int = 64,
        mlp_hidden_dims: list = None,
        dropout: float = 0.0,
        use_sigmoid: bool = True,
    ):
        super().__init__()
        if mlp_hidden_dims is None:
            mlp_hidden_dims = [128, 64]

        # GMF branch
        self.user_emb_gmf = nn.Embedding(num_users, emb_dim)
        self.item_emb_gmf = nn.Embedding(num_items, emb_dim)

        # MLP branch
        self.user_emb_mlp = nn.Embedding(num_users, emb_dim)
        self.item_emb_mlp = nn.Embedding(num_items, emb_dim)

        mlp_layers = []
        input_dim = emb_dim * 2
        for h in mlp_hidden_dims:
            mlp_layers.append(nn.Linear(input_dim, h))
            mlp_layers.append(nn.ReLU())
            if dropout > 0:
                mlp_layers.append(nn.Dropout(dropout))
            input_dim = h

        self.mlp = nn.Sequential(*mlp_layers)

        # Output layer
        self.fc = nn.Linear(emb_dim + mlp_hidden_dims[-1], 1)
        self.use_sigmoid = use_sigmoid

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, user, item):
        gmf_out = self.user_emb_gmf(user) * self.item_emb_gmf(item)
        mlp_in = torch.cat([self.user_emb_mlp(user), self.item_emb_mlp(item)], dim=1)
        mlp_out = self.mlp(mlp_in)
        concat = torch.cat([gmf_out, mlp_out], dim=1)
        logits = self.fc(concat).squeeze(-1)
        if self.use_sigmoid:
            return torch.sigmoid(logits)
        return logits

In [ ]:
def prepare_val_df(
    df_val_raw,
    user2idx,
    item2idx,
):
    df = df_val_raw.copy()
    df["user_idx"] = df["user_id"].map(user2idx)
    df["item_idx"] = df["tv_show_id"].map(item2idx)
    df = df[df["user_idx"].notna() & df["item_idx"].notna()]
    df["user_idx"] = df["user_idx"].astype(int)
    df["item_idx"] = df["item_idx"].astype(int)
    return df
def aggregate_interactions(df):
    df_sess = (
        df
        .groupby(["user_idx", "item_idx", "session_id"])
        .agg(
            screen_time=("screen_time", "sum"),
            duration_view=("duration_view", "sum"),
        )
        .reset_index()
    )
    df_agg = (
        df_sess
        .groupby(["user_idx", "item_idx"])
        .agg(
            total_screen_time=("screen_time", "sum"),
            avg_screen_time=("screen_time", "mean"),
            view_count=("session_id", "nunique"),
        )
        .reset_index()
    )

    return df_agg

def build_preference_label(df_agg, alpha=0.7, beta=0.3, threshold=None):
    df = df_agg.copy()

    df["screen_norm"] = np.log1p(df["total_screen_time"])
    df["view_norm"] = np.log1p(df["view_count"])

    df["preference_score"] = (
        alpha * df["screen_norm"] +
        beta * df["view_norm"]
    )

    if threshold is not None:
        df["label"] = (df["preference_score"] >= threshold).astype(int)

    return df



def filter_val_users(
    df_val,
    train_users,
    min_pos_items=5,
):
    df = df_val[df_val["user_idx"].isin(train_users)]

    pos_cnt = (
        df[df["label"] == 1]
        .groupby("user_idx")["item_idx"]
        .nunique()
    )

    valid_users = pos_cnt[pos_cnt >= min_pos_items].index

    df = df[df["user_idx"].isin(valid_users)]

    return df


In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0.0
    for u, i, l in loader:
        u, i, l = u.to(device), i.to(device), l.to(device)
        optimizer.zero_grad()
        pred = model(u, i).squeeze()
        loss = criterion(pred, l)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(l)
    return total_loss / len(loader.dataset)


def evaluate_metrics(model, df_val, n_items, popular_items, k=5, device='cuda'):
    """Evaluate model on validation set"""
    model.eval()
    recalls = []
    maps = []
    ndcgs = []
    
    user_groups = df_val.groupby('user_idx')
    with torch.no_grad():
        for u, group in user_groups:
            true_items = set(group[group['label'] == 1]['item_idx'].values)
            if len(true_items) == 0:
                continue
            
            users = torch.full((n_items,), u, dtype=torch.long, device=device)
            items = torch.arange(n_items, dtype=torch.long, device=device)
            scores = model(users, items).squeeze().cpu().numpy()
            top_idx = np.argsort(scores)[-k:][::-1]
            
            hit = len(true_items.intersection(set(top_idx)))
            recalls.append(hit / len(true_items))
            
            rel = [1 if i in true_items else 0 for i in top_idx]
            if sum(rel) == 0:
                maps.append(0.0)
            else:
                hit_cnt = 0
                precisions = []
                for idx, r in enumerate(rel):
                    if r == 1:
                        hit_cnt += 1
                        precisions.append(hit_cnt / (idx + 1))
                maps.append(np.mean(precisions))
            ndcgs.append(ndcg_at_k(rel, k))
    
    return {
        f'Recall@{k}': float(np.mean(recalls)) if recalls else 0.0,
        f'MAP@{k}': float(np.mean(maps)) if maps else 0.0,
        f'NDCG@{k}': float(np.mean(ndcgs)) if ndcgs else 0.0,
    }

# Prepare train/val splits for hyperparameter tuning
df_train_clean_indexed = prepare_val_df(df_train_clean, userid_to_useridx, showid_to_itemidx)
df_val_clean_indexed = prepare_val_df(df_val_clean, userid_to_useridx, showid_to_itemidx)

print(f"Train indexed: {len(df_train_clean_indexed)}")
print(f"Val indexed: {len(df_val_clean_indexed)}")

In [ ]:
# Aggregate interactions from train set
df_train_agg = aggregate_interactions(df_train_clean_indexed)

# Hyperparameter search space
hparams_grid = {
    'alpha': [0.6, 0.7, 0.8],
    'beta': [0.2, 0.3, 0.4],
    'threshold': [0.2, 0.3, 0.4],
    'lr': [0.001, 0.0005],
    'batch_size': [512, 1024],
    'epochs': [5, 7],
    'emb_dim': [64],
}

print("=" * 60)
print("HYPERPARAMETER TUNING ON VALIDATION SET")
print("=" * 60)

best_score = 0.0
best_hparams = None
tune_results = []

# Create grid of all combinations
from itertools import product
param_names = list(hparams_grid.keys())
param_values = [hparams_grid[name] for name in param_names]

trial_count = 0
for param_combo in product(*param_values):
    hparams = dict(zip(param_names, param_combo))
    trial_count += 1
    
    # Build preference labels with current alpha, beta, threshold
    df_train_labeled = build_preference_label(
        df_train_agg,
        alpha=hparams['alpha'],
        beta=hparams['beta'],
        threshold=hparams['threshold'],
    )
    
    # Prepare negative samples
    user_item_set = df_train_agg.groupby('user_idx')['item_idx'].apply(set).to_dict()
    neg_samples = []
    for u in user_item_set:
        n_pos = len(user_item_set[u])
        neg_items = np.random.choice(
            list(set(range(n_items)) - user_item_set[u]),
            size=n_pos * 4, replace=True
        )
        for i in neg_items:
            neg_samples.append([u, i, 0])
    
    df_neg = pd.DataFrame(neg_samples, columns=['user_idx', 'item_idx', 'label'])
    df_train_final = pd.concat([
        df_train_labeled[['user_idx', 'item_idx', 'label']], df_neg
    ]).reset_index(drop=True)
    
    # Create data loader
    train_dataset = InteractionDataset(df_train_final)
    train_loader = DataLoader(
        train_dataset,
        batch_size=hparams['batch_size'],
        shuffle=True
    )
    
    # Train model
    model = NeuMF(n_users, n_items, emb_dim=hparams['emb_dim']).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=hparams['lr'])
    criterion = nn.BCELoss()
    
    for epoch in range(hparams['epochs']):
        train_epoch(model, train_loader, optimizer, criterion, device)
    
    # Evaluate on validation set
    df_val_labeled = build_preference_label(
        aggregate_interactions(df_val_clean_indexed),
        alpha=hparams['alpha'],
        beta=hparams['beta'],
        threshold=hparams['threshold'],
    )
    
    # Filter validation users that appear in training
    train_users = set(df_train_clean_indexed['user_idx'].unique())
    df_val_filtered = df_val_labeled[df_val_labeled['user_idx'].isin(train_users)].copy()
    
    pos_cnt = (df_val_filtered[df_val_filtered['label'] == 1]
              .groupby('user_idx')['item_idx'].nunique())
    valid_users = pos_cnt[pos_cnt >= 5].index
    df_val_filtered = df_val_filtered[df_val_filtered['user_idx'].isin(valid_users)]
    
    if len(df_val_filtered) > 0:
        metrics = evaluate_metrics(model, df_val_filtered, n_items, [], k=5, device=device)
        score = metrics['NDCG@5']
    else:
        score = 0.0
    
    tune_results.append({
        'trial': trial_count,
        'hparams': hparams,
        'NDCG@5': score,
        'Recall@5': metrics.get('Recall@5', 0.0),
        'MAP@5': metrics.get('MAP@5', 0.0),
    })
    
    print(f"Trial {trial_count:3d} | NDCG@5: {score:.4f} | "
          f"α={hparams['alpha']}, β={hparams['beta']}, th={hparams['threshold']}, "
          f"lr={hparams['lr']}, bs={hparams['batch_size']}, ep={hparams['epochs']}")
    
    if score > best_score:
        best_score = score
        best_hparams = hparams.copy()
        print(f"    ✓ NEW BEST! NDCG@5={best_score:.4f}")

print("\n" + "=" * 60)
print(f"BEST HYPERPARAMETERS (NDCG@5={best_score:.4f})")
print("=" * 60)
for key, val in best_hparams.items():
    print(f"  {key}: {val}")
print("=" * 60)


print("\n" + "=" * 60)
print("FINAL TRAINING ON MERGED DATA")
print("=" * 60)

# Aggregate interactions from merged data
df_merged_indexed = prepare_val_df(df_merged, userid_to_useridx, showid_to_itemidx)
df_final_agg = aggregate_interactions(df_merged_indexed)

# Build preference labels with best hyperparameters
df_final_labeled = build_preference_label(
    df_final_agg,
    alpha=best_hparams['alpha'],
    beta=best_hparams['beta'],
    threshold=best_hparams['threshold'],
)

print(f"✓ Aggregated interactions: {len(df_final_agg)}")
print(f"✓ Labeled interactions: {len(df_final_labeled)}")
print(f"✓ Positive samples: {(df_final_labeled['label'] == 1).sum()}")

# Prepare negative samples
user_item_set_final = df_final_agg.groupby('user_idx')['item_idx'].apply(set).to_dict()
neg_samples_final = []
for u in user_item_set_final:
    n_pos = len(user_item_set_final[u])
    neg_items = np.random.choice(
        list(set(range(n_items)) - user_item_set_final[u]),
        size=n_pos * 4, replace=True
    )
    for i in neg_items:
        neg_samples_final.append([u, i, 0])

df_neg_final = pd.DataFrame(neg_samples_final, columns=['user_idx', 'item_idx', 'label'])
df_train_final = pd.concat([
    df_final_labeled[['user_idx', 'item_idx', 'label']], df_neg_final
]).reset_index(drop=True)

print(f"✓ Final training set: {len(df_train_final)}")

# Create data loader
train_dataset_final = InteractionDataset(df_train_final)
train_loader_final = DataLoader(
    train_dataset_final,
    batch_size=best_hparams['batch_size'],
    shuffle=True
)

# Build final model
model_final = NeuMF(n_users, n_items, emb_dim=best_hparams['emb_dim']).to(device)
optimizer_final = torch.optim.Adam(model_final.parameters(), lr=best_hparams['lr'])
criterion_final = nn.BCELoss()

# Train final model
loss_history = []
for epoch in range(best_hparams['epochs']):
    loss = train_epoch(model_final, train_loader_final, optimizer_final, criterion_final, device)
    loss_history.append(loss)
    print(f"  Epoch {epoch + 1}/{best_hparams['epochs']}: Loss={loss:.4f}")

# Save model
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = f'models/neumf_final_{timestamp}.pth'
torch.save(model_final.state_dict(), model_path)
print(f"✓ Model saved: {model_path}")

# Plot loss curve
plt.figure(figsize=(10, 5))
plt.plot(range(1, best_hparams['epochs'] + 1), loss_history, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss (NeuMF - Final)')
plt.grid(True)
plt.tight_layout()
plt.savefig(f'results/loss_curve_{timestamp}.png')
plt.close()
print(f"✓ Loss curve saved")

# Calculate and display item popularity for cold start
item_popularity = (
    df_final_labeled.groupby('item_idx')['preference_score'].mean()
)
popular_items = item_popularity.sort_values(ascending=False).index.tolist()
print(f"✓ Top 5 popular items: {popular_items[:5]}")

print("\n" + "=" * 60)
print("INFERENCE ON TEST SET")
print("=" * 60)


In [ ]:
def predict_topk(
    model,
    df_submission,
    n_users,
    n_items,
    popular_items,
    allowed_items,
    itemidx_to_showid,
    k=5,
    device='cuda',
):
    """Generate top-k predictions for submission"""
    model.eval()
    result = {}
    allowed_items = np.array(list(allowed_items))

    with torch.no_grad():
        for idx, row in df_submission.iterrows():
            user_id = str(row['user_id'])
            user_idx = row.get('user_idx')

            # Cold start: user not in training
            if pd.isna(user_idx) or user_idx < 0 or user_idx >= n_users:
                top_idx = popular_items[:k]
            else:
                u = torch.full(
                    (len(allowed_items),),
                    int(user_idx),
                    dtype=torch.long,
                    device=device,
                )
                i = torch.tensor(allowed_items, dtype=torch.long, device=device)
                scores = model(u, i).squeeze().cpu().numpy()
                top_idx = allowed_items[np.argsort(scores)[-k:][::-1]]

            # Map item_idx → tv_show_id
            result[user_id] = [int(itemidx_to_showid[int(x)]) for x in top_idx]

    return result


# Load submission template (from metadata test)
df_test_metadata_clean = df_metadata_test_clean.copy()
df_test_metadata_clean['tv_show_id'] = df_test_metadata_clean['tv_show_id'].astype(str)
df_test_metadata_clean['item_idx'] = (
    df_test_metadata_clean['tv_show_id'].map(showid_to_itemidx)
)
df_test_metadata_clean = df_test_metadata_clean.dropna(subset=['item_idx'])

# Create submission dataframe
df_submission = pd.DataFrame({
    'user_id': sorted(df_test_metadata_clean['user_id'].unique())
})

# Map user indices
df_submission['user_idx'] = df_submission['user_id'].map(userid_to_useridx)

print(f"✓ Submission users: {len(df_submission)}")
print(f"✓ Available test items: {len(df_test_metadata_clean)}")

# Get allowed items for filtering
allowed_items = set(df_test_metadata_clean['item_idx'].astype(int).unique())

# Generate predictions
predictions = predict_topk(
    model=model_final,
    df_submission=df_submission,
    n_users=n_users,
    n_items=n_items,
    popular_items=popular_items,
    allowed_items=allowed_items,
    itemidx_to_showid=itemidx_to_showid,
    k=5,
    device=device,
)

print(f"✓ Generated {len(predictions)} predictions")
print(f"✓ Sample prediction: {list(predictions.items())[0]}")

# Export submission
def export_submission(pred_dict, save_path='submission_final.csv'):
    """Export predictions to CSV"""
    rows = []
    for user_id, shows in pred_dict.items():
        rows.append({
            'user_id': user_id,
            'tv_show_id': ' '.join(map(str, shows))
        })
    df_out = pd.DataFrame(rows)
    df_out.to_csv(save_path, index=False)
    print(f"✓ Submission saved: {save_path}")
    return df_out

df_submission_final = export_submission(predictions, 'submission_final.csv')

# Validation checks
assert len(predictions) == len(df_submission), "Predictions count mismatch"
assert all(len(v) == 5 for v in predictions.values()), "Not all predictions have 5 items"
all_items = set(df_test_metadata_clean['tv_show_id'].astype(str).unique())
assert all(
    str(i) in all_items
    for v in predictions.values()
    for i in v
), "Invalid items in predictions"

print("\n✓ All validation checks passed!")
print(df_submission_final.head())


print("\n" + "=" * 80)
print("PIPELINE SUMMARY")
print("=" * 80)

summary = {
    'timestamp': timestamp,
    'data_statistics': {
        'total_users': n_users,
        'total_items': n_items,
        'train_interactions': len(df_train_clean),
        'val_interactions': len(df_val_clean),
        'total_interactions': len(df_merged),
    },
    'best_hyperparameters': best_hparams,
    'tuning_trials': len(tune_results),
    'best_val_metrics': {
        'NDCG@5': float(best_score),
    },
    'final_model': {
        'epochs_trained': best_hparams['epochs'],
        'final_loss': float(loss_history[-1]),
    },
    'submission': {
        'total_predictions': len(predictions),
        'items_per_user': 5,
    }
}

# Print summary
print("\n📊 DATA STATISTICS")
print(f"  Users: {summary['data_statistics']['total_users']}")
print(f"  Items: {summary['data_statistics']['total_items']}")
print(f"  Train interactions: {summary['data_statistics']['train_interactions']}")
print(f"  Val interactions: {summary['data_statistics']['val_interactions']}")
print(f"  Total interactions: {summary['data_statistics']['total_interactions']}")

print("\n🔧 BEST HYPERPARAMETERS")
for key, val in best_hparams.items():
    print(f"  {key}: {val}")

print(f"\n📈 TUNING RESULTS")
print(f"  Trials: {summary['tuning_trials']}")
print(f"  Best NDCG@5: {summary['best_val_metrics']['NDCG@5']:.4f}")

print(f"\n🎓 FINAL TRAINING")
print(f"  Epochs: {summary['final_model']['epochs_trained']}")
print(f"  Final Loss: {summary['final_model']['final_loss']:.4f}")

print(f"\n📤 SUBMISSION")
print(f"  Total predictions: {summary['submission']['total_predictions']}")
print(f"  Items per user: {summary['submission']['items_per_user']}")
print(f"  Output file: submission_final.csv")

print("\n" + "=" * 80)

# Save summary
with open(f'results/pipeline_summary_{timestamp}.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✓ Summary saved to results/pipeline_summary_{timestamp}.json")

# Save tuning results
df_tune_results = pd.DataFrame(tune_results)
df_tune_results.to_csv(f'results/tuning_results_{timestamp}.csv', index=False)
print(f"✓ Tuning results saved to results/tuning_results_{timestamp}.csv")

print("\n" + "=" * 80)
print("✓ PIPELINE COMPLETED SUCCESSFULLY!")
print("=" * 80)